# 08 — Reshaping

Cambiar la forma del DataFrame: de ancho a largo, de largo a ancho, transponer, explotar listas.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)


## melt() — de ancho a largo

Convierte columnas en filas. Útil cuando los datos tienen una columna por período (ene, feb, mar…) y se necesita una columna `mes` y una columna `valor`.

In [ ]:
# Crear un DataFrame ancho de ejemplo — ventas por región y trimestre
ventas_wide = pd.DataFrame({
    'region': ['West', 'East', 'South', 'Central'],
    'Q1':     [240000, 185000, 98000, 130000],
    'Q2':     [270000, 210000, 112000, 145000],
    'Q3':     [225000, 195000, 105000, 138000],
    'Q4':     [310000, 230000, 120000, 160000],
})
print('Formato ancho:')
print(ventas_wide)

# melt: id_vars = columnas que se mantienen, value_vars = columnas que se convierten en filas
ventas_long = ventas_wide.melt(
    id_vars='region',
    value_vars=['Q1', 'Q2', 'Q3', 'Q4'],
    var_name='trimestre',
    value_name='ventas'
)
print()
print('Formato largo:')
print(ventas_long.sort_values(['region', 'trimestre']))


## pivot() — de largo a ancho

El opuesto de melt. Requiere que la combinación de index + columns sea única.

In [ ]:
# Invertir el melt anterior
ventas_recuperada = ventas_long.pivot(
    index='region',
    columns='trimestre',
    values='ventas'
).reset_index()

ventas_recuperada.columns.name = None   # quitar el nombre del eje de columnas
print(ventas_recuperada)


## pivot_table() — pivot con agregación

Cuando hay duplicados en la combinación index+columns, `pivot()` falla. `pivot_table()` los resuelve con una función de agregación.

In [ ]:
# Ventas totales por Region y Category
tabla = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
).round(0)
print(tabla)
print()

# Múltiples valores de agregación
tabla2 = pd.pivot_table(
    df,
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc=['sum', 'count']
)
print(tabla2.head(2))


## crosstab() — tabla de frecuencias

In [ ]:
# Frecuencia de combinaciones entre dos columnas categóricas
tabla = pd.crosstab(
    df['Region'],
    df['Category'],
    margins=True
)
print(tabla)
print()

# normalize='index' para proporciones por fila
tabla_pct = pd.crosstab(
    df['Region'],
    df['Category'],
    normalize='index'
).round(3)
print(tabla_pct)


## explode() — expandir listas en filas

Cuando una celda contiene una lista, `explode()` crea una fila por elemento de la lista.

In [ ]:
# Simular una columna con listas (como amenities en Airbnb)
df_amenities = air[['listing_id', 'amenities']].head(5).copy()
df_amenities['amenities'] = df_amenities['amenities'].str.split(',')
print('Antes:')
print(df_amenities)

# explode — una fila por amenity
df_exploded = df_amenities.explode('amenities')
df_exploded['amenities'] = df_exploded['amenities'].str.strip().str.strip('"[]')
print()
print('Después:')
print(df_exploded.head(15))


---
## Resumen

| Operación | Sintaxis | Caso de uso |
|-----------|----------|-------------|
| Columnas → filas | `df.melt(id_vars, value_vars)` | Normalizar datos anchos |
| Filas → columnas | `df.pivot(index, columns, values)` | Sin duplicados |
| Filas → columnas (con agg) | `pd.pivot_table(df, values, index, columns, aggfunc)` | Con duplicados |
| Frecuencias cruzadas | `pd.crosstab(col1, col2)` | Análisis categórico |
| Lista → filas | `df.explode('col')` | Columnas con listas |
